# Fracture Detection using DETR Transformer
> Medical Imaging AI | MSCS SJSU | PyTorch · DETR · ONNX

This notebook covers:
1. Dataset loading (DICOM + COCO format)
2. DETR model training with PyTorch Lightning
3. Clinical metrics (AUC, Sensitivity, Specificity)
4. GradCAM visualization
5. ONNX export + parity test + benchmark

## 1. Install Dependencies

In [ ]:
!pip install -q supervision==0.3.0 transformers pytorch-lightning timm
!pip install -q pycocotools scipy pydicom opencv-python scikit-learn
!pip install -q onnx onnxruntime pandas matplotlib

## 2. Imports & Setup

In [ ]:
import os
import sys
import random
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import supervision as sv
import pytorch_lightning as pl
from transformers import DetrImageProcessor, DetrForObjectDetection

# Add src to path
sys.path.insert(0, '../src')

print(f"PyTorch:     {torch.__version__}")
print(f"Lightning:   {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

## 3. Dataset — COCO with DICOM Support

In [ ]:
from dataset import CocoDetection, build_dataloaders, load_image

DATASET_ROOT = '../data/bone fracture.v2-release.coco'
image_processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')

loaders, train_ds, val_ds, test_ds = build_dataloaders(
    DATASET_ROOT, image_processor, batch_size=4, num_workers=0
)

# Build label maps
categories = train_ds.coco.cats
id2label = {k: v['name'] for k, v in categories.items()}
label2id = {v: k for k, v in id2label.items()}
print(f"Labels: {id2label}")

### 3a. Visualize a training sample

In [ ]:
# Pick a random image from the training set
image_ids = train_ds.coco.getImgIds()
image_id  = random.choice(image_ids)
img_meta  = train_ds.coco.loadImgs(image_id)[0]
img_path  = os.path.join(train_ds.root, img_meta['file_name'])

# load_image handles both .jpg and .dcm
pil_img = load_image(img_path)
annotations = train_ds.coco.imgToAnns[image_id]

img_np = np.array(pil_img)
detections = sv.Detections.from_coco_annotations(coco_annotation=annotations)
labels = [id2label[cls_id] for _, _, cls_id, _ in detections]

box_annotator = sv.BoxAnnotator()
annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections, labels=labels)

plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.title(f'Image #{image_id} — {len(annotations)} annotation(s)')
plt.axis('off')
plt.show()

### 3b. DICOM Loading Demo

In [ ]:
# Demo: load a DICOM file directly (replace path with your .dcm file)
# dcm_path = '../data/sample.dcm'
# pil_img = load_image(dcm_path)
# plt.imshow(pil_img, cmap='gray')
# plt.title('DICOM X-Ray')
# plt.axis('off')
# plt.show()
print('Replace dcm_path with your DICOM file to test DICOM loading')

## 4. Model — DETR PyTorch Lightning

In [ ]:
from model import DETRFractureDetector

model = DETRFractureDetector(
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    lr=1e-4,
    lr_backbone=1e-5,
)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Total parameters:     {total_params:.1f}M")
print(f"Trainable parameters: {trainable:.1f}M")

## 5. Training

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_cb = ModelCheckpoint(
    dirpath='../results/checkpoints',
    filename='detr-fracture-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=3,
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, mode='min')

trainer = pl.Trainer(
    max_epochs=30,
    callbacks=[checkpoint_cb, early_stop],
    log_every_n_steps=5,
    accelerator='auto',
    devices=1,
)

trainer.fit(model, loaders['train'], loaders['val'])

## 6. Clinical Metrics (Sensitivity, Specificity, AUC)

In [ ]:
from evaluate import evaluate_model, print_metrics_table

# Load best checkpoint
best_ckpt = checkpoint_cb.best_model_path
best_model = DETRFractureDetector.load_from_checkpoint(
    best_ckpt, num_labels=len(id2label), id2label=id2label, label2id=label2id
)

metrics = evaluate_model(
    model=best_model.model,
    dataloader=loaders['test'],
    image_processor=image_processor,
    iou_threshold=0.5,
    score_threshold=0.5,
    device=DEVICE,
)
print_metrics_table(metrics)

## 7. GradCAM Visualization

In [ ]:
from gradcam import run_gradcam

# Pick a test image
test_ids = test_ds.coco.getImgIds()
test_img_id = test_ids[0]
test_img_meta = test_ds.coco.loadImgs(test_img_id)[0]
test_img_path = os.path.join(test_ds.root, test_img_meta['file_name'])

out_path, heatmap, results = run_gradcam(
    image_path=test_img_path,
    model_or_checkpoint=best_model.model,
    image_processor=image_processor,
    output_dir='../results/gradcam',
    score_threshold=0.3,
    id2label=id2label,
)

# Display inline
grad_img = plt.imread(out_path)
plt.figure(figsize=(18, 6))
plt.imshow(grad_img)
plt.axis('off')
plt.title('GradCAM — Original | Heatmap | Detection')
plt.show()
print(f'Detected {len(results["boxes"])} fracture(s)')

## 8. ONNX Export

In [ ]:
import sys
sys.path.insert(0, '../export')
from export_onnx import export_to_onnx, verify_onnx

# Export FP32
onnx_fp32 = export_to_onnx(
    model=best_model.model,
    output_path='../results/detr_fracture_fp32.onnx',
    image_size=800,
    use_fp16=False,
)
verify_onnx(onnx_fp32)

# Export FP16
onnx_fp16 = export_to_onnx(
    model=best_model.model,
    output_path='../results/detr_fracture_fp16.onnx',
    image_size=800,
    use_fp16=True,
)
verify_onnx(onnx_fp16)

## 9. Parity Test (PyTorch vs ORT ≤ 1e-4)

In [ ]:
from parity_test import parity_check

passed = parity_check(
    onnx_path=onnx_fp32,
    pytorch_model=best_model.model,
    image_size=800,
    batch_sizes=[1, 4],
    atol=1e-4,
)
print('Parity test passed!' if passed else 'Parity test FAILED — check export')

## 10. Benchmark (Latency: PyTorch vs ORT at FP32/FP16)

In [ ]:
from benchmark import run_benchmark

rows = run_benchmark(
    pytorch_model=best_model.model,
    onnx_fp32=onnx_fp32,
    onnx_fp16=onnx_fp16,
    batch_sizes=[1, 4, 8, 16],
    output_dir='../results',
)

# Display chart
chart = plt.imread('../results/benchmark_chart.png')
plt.figure(figsize=(14, 6))
plt.imshow(chart)
plt.axis('off')
plt.show()

## Summary

| Component | Status |
|---|---|
| DICOM input support | ✅ pydicom |
| GradCAM heatmap | ✅ backbone.layer4 hooks |
| Clinical metrics (AUC, Sensitivity, Specificity) | ✅ evaluate.py |
| ONNX export (FP32 + FP16) | ✅ export_onnx.py |
| Parity test (PyTorch vs ORT ≤1e-4) | ✅ parity_test.py |
| Latency benchmark | ✅ benchmark_results.csv |
| HuggingFace Gradio demo | 🔜 Week 3 |